# BAAI/bge-m3 Embedding Generation

This notebook generates dense embeddings for all WikiCS nodes using the [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3) model.

**Features:** Incremental saving — progress is saved after every batch and already-embedded nodes are skipped on restart.

**Output:** `master_embeddings.parquet` with columns `id` (node id) and `embedding` (dense vector).

In [5]:
import json
import os
import torch
import pandas as pd
from FlagEmbedding import BGEM3FlagModel

In [6]:
# Load the dataset
with open("../../data/data-embeddings.json", "r") as f:
    data = json.load(f)

nodes = data["nodes"]
print(f"Total nodes: {len(nodes)}")

Total nodes: 11701


In [7]:
# Quick analysis: how long are the texts? (rough token estimate ~4 chars/token)
import numpy as np
lengths = np.array([len(n["raw_text"]) for n in nodes]) / 4

print(f"Estimated token lengths across {len(nodes)} nodes:")
for p in [50, 75, 90, 95, 99]:
    print(f"  P{p}: {int(np.percentile(lengths, p)):>6} tokens")
print(f"  Max: {int(np.max(lengths)):>6} tokens")
print(f"\nNodes truncated at 1024: {int(np.sum(lengths > 1024))} ({np.sum(lengths > 1024)/len(nodes)*100:.1f}%)")
print(f"Nodes truncated at 2048: {int(np.sum(lengths > 2048))} ({np.sum(lengths > 2048)/len(nodes)*100:.1f}%)")

Estimated token lengths across 11701 nodes:
  P50:    706 tokens
  P75:   1442 tokens
  P90:   2952 tokens
  P95:   4618 tokens
  P99:   9940 tokens
  Max:  36388 tokens

Nodes truncated at 1024: 4253 (36.3%)
Nodes truncated at 2048: 1934 (16.5%)


In [8]:
# Load existing progress if available
OUTPUT_FILE = "master_embeddings.parquet"

if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_parquet(OUTPUT_FILE)
    done_ids = set(df_existing["id"].tolist())
    print(f"Resuming: {len(done_ids)} nodes already embedded, {len(nodes) - len(done_ids)} remaining.")
else:
    df_existing = pd.DataFrame(columns=["id", "embedding"])
    done_ids = set()
    print("Starting fresh — no existing embeddings found.")

# Filter to only the nodes that still need embedding
remaining_nodes = [n for n in nodes if n["id"] not in done_ids]
print(f"Nodes to embed: {len(remaining_nodes)}")

Starting fresh — no existing embeddings found.
Nodes to embed: 11701


In [9]:
# Load the BAAI/bge-m3 model
# use_fp16=True to fit within 8GB VRAM
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda")
print("Model loaded successfully.")

Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


Model loaded successfully.


In [10]:
# Generate embeddings in batches with incremental saving
BATCH_SIZE = 128
MAX_LENGTH = 2048
SAVE_EVERY = 10  # save to disk every N batches

# Sort remaining nodes by text length so similarly-sized texts are batched together
# This minimises padding waste and dramatically speeds up encoding
remaining_nodes_sorted = sorted(remaining_nodes, key=lambda n: len(n["raw_text"]))

remaining_ids = [n["id"] for n in remaining_nodes_sorted]
remaining_texts = [n["raw_text"] for n in remaining_nodes_sorted]
total_batches = (len(remaining_texts) + BATCH_SIZE - 1) // BATCH_SIZE

rows = df_existing.to_dict("records")

for i in range(0, len(remaining_texts), BATCH_SIZE):
    batch_num = i // BATCH_SIZE + 1
    batch_texts = remaining_texts[i : i + BATCH_SIZE]
    batch_ids = remaining_ids[i : i + BATCH_SIZE]

    output = model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )

    for node_id, emb in zip(batch_ids, output["dense_vecs"]):
        rows.append({"id": node_id, "embedding": emb.tolist()})

    # Save incrementally every SAVE_EVERY batches (and always on the last batch)
    if batch_num % SAVE_EVERY == 0 or batch_num == total_batches:
        df_save = pd.DataFrame(rows)
        df_save.to_parquet(OUTPUT_FILE, index=False)
        print(f"Batch {batch_num}/{total_batches} done — {len(rows)} total embeddings saved. [SAVED]")
    else:
        print(f"Batch {batch_num}/{total_batches} done — {len(rows)} total embeddings.")

print(f"\nFinished! {len(rows)} embeddings saved to {OUTPUT_FILE}")

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 250.06it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 23.26it/s]


Batch 1/92 done — 128 total embeddings.


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 14.49it/s]


Batch 2/92 done — 256 total embeddings.


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 14.71it/s]


...

Batch 89/92 done — 11392 total embeddings.


Inference Embeddings: 100%|██████████| 1/1 [06:30<00:00, 390.07s/it]


Batch 90/92 done — 11520 total embeddings saved. [SAVED]


Inference Embeddings: 100%|██████████| 1/1 [06:30<00:00, 390.08s/it]


Batch 91/92 done — 11648 total embeddings.
Batch 92/92 done — 11701 total embeddings saved. [SAVED]

Finished! 11701 embeddings saved to master_embeddings.parquet


In [ ]:
# Verify the saved file
df_check = pd.read_parquet(OUTPUT_FILE)
print(f"Verified: {len(df_check)} rows, embedding dim = {len(df_check.iloc[0]['embedding'])}")
df_check.head()